# RCA Pipeline (Optimized)

Clean rebuild with schema-aware joins, faster labeling, and clear diagnostics.

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import mysql.connector
from sklearn.preprocessing import LabelEncoder

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 180)

def norm_ckt(v):
    if pd.isna(v):
        return None
    s = str(v).strip()
    try:
        x = float(s)
        return str(int(x)) if x == int(x) else s
    except Exception:
        return s

def to_dt(df, cols):
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_datetime(df[c], errors='coerce')
    return df


In [ ]:
# 1) Connect + load only required tables/columns
conn = mysql.connector.connect(host='localhost', user='root', password='Alok@12&kumar#@', database='railtel')

TABLE_COLS = {
    'alarm': [
        'ALARM_ID_PK','OPEN_TIME','CLOSURE_TIME','SEVERITY','ACTUAL_SEVERITY','ALARM_STATUS','ALARM_CODE','ALARM_NAME',
        'EVENT_TYPE','ENTITY_NAME','ENTITY_ID','PARENT_ENTITY_ID','ENTITY_TYPE','VENDOR','TECHNOLOGY','DOMAIN','PROBABLE_CAUSE',
        'CORRELATION_TYPE','CORRELATION_FLAG','INCIDENT_ID','PARTICIPATED_IN_INCIDENT','INCIDENT_STATUS','GEOGRAPHY_L1_NAME',
        'GEOGRAPHY_L2_NAME','GEOGRAPHY_L3_NAME','GEOGRAPHY_L4_NAME','CKT_ID','SERVICE_AFFECTED','DESCRIPTION'
    ],
    'alarm_library': ['ALARM_IDENTIFIER','ALARM_ID','CLIENT_ALARM_CODE','ALARM_NAME','ALARM_HIERARCHY','CORRELATION_ENABLE','CONTRIBUTOR_CATEGORY','PROBABLE_CAUSE'],
    'network_element': ['ID','NE_ID','NE_NAME','NE_TYPE','VENDOR','TECHNOLOGY','DOMAIN','NE_STATUS','IPV4','IPV6'],
    'ckt_id_info': ['ID','NETWORK_ELEMENT_ID_FK','CKT_ID','IS_DELETED','CREATED_TIME'],
    'network_service_history': ['ID','NS_ID','STATUS','CKT_ID','MODIFIED_TIME','SOURCE_ROUTER_IP'],
    'lsp': ['ID','LSP_NAME','SOURCE_NE_ID','DESTINATION_NE_ID','SOURCE_IP','DESTINATION_IP'],
    'lsp_hop': ['ID','LSP_ID','HOP_SEQUENCE','ROUTER_NE_ID','INTERFACE_NE_ID'],
    'bgp_link': ['SOURCE_NE_ID','DESTINATION_NE_ID'],
    'isis_link': ['SOURCE_NE_ID','DESTINATION_NE_ID'],
    'ospf_link': ['SOURCE_NE_ID','DESTINATION_NE_ID'],
    'lldp_link': ['SOURCE_INTERFACE_NE_ID','DESTINATION_INTERFACE_NE_ID']
}

tables = pd.read_sql('SHOW TABLES', conn).iloc[:,0].tolist()
data = {}
for t, cols in TABLE_COLS.items():
    if t not in tables:
        data[t] = pd.DataFrame()
        continue
    df = pd.read_sql(f'SELECT * FROM {t}', conn)
    keep = [c for c in cols if c in df.columns]
    data[t] = df[keep].copy()

print({k: v.shape for k,v in data.items()})

In [ ]:
# 2) Clean + canonical keys
alarm = data['alarm'].copy()
ne = data['network_element'].copy()
alib = data['alarm_library'].copy()
ckt = data['ckt_id_info'].copy()
ns_hist = data['network_service_history'].copy()
lsp_hop = data['lsp_hop'].copy()

alarm = to_dt(alarm, ['OPEN_TIME','CLOSURE_TIME'])
ns_hist = to_dt(ns_hist, ['MODIFIED_TIME'])

alarm['ENTITY_ID_STR'] = alarm['ENTITY_ID'].astype(str)
alarm['CKT_N'] = alarm['CKT_ID'].map(norm_ckt)
alarm = alarm.dropna(subset=['ALARM_ID_PK','OPEN_TIME','ENTITY_ID'])

if 'NE_STATUS' in ne.columns and 'STATUS' not in ne.columns:
    ne['STATUS'] = ne['NE_STATUS']
ne['NE_ID'] = ne['NE_ID'].astype(str)

id_to_neid = ne.set_index('ID')['NE_ID'].to_dict() if 'ID' in ne.columns else {}
neid_to_name = ne.set_index('NE_ID')['NE_NAME'].to_dict() if 'NE_NAME' in ne.columns else {}

print('alarm rows:', len(alarm), '| ne rows:', len(ne))

In [ ]:
# 3) Method 1: Alarm library root logic (strict + relaxed fallback)
labels_m1 = pd.DataFrame(columns=['INCIDENT_ID','ROOT_NE_ID','ROOT_NE_NAME','LABEL_SOURCE'])

if not alib.empty:
    h = alib['ALARM_HIERARCHY'].astype(str).str.upper() if 'ALARM_HIERARCHY' in alib.columns else ''
    c = alib['CONTRIBUTOR_CATEGORY'].astype(str).str.upper() if 'CONTRIBUTOR_CATEGORY' in alib.columns else ''
    p = alib['PROBABLE_CAUSE'].astype(str).str.upper() if 'PROBABLE_CAUSE' in alib.columns else ''
    base_mask = h.str.contains('ROOT|PRIMARY', na=False) | c.str.contains('ROOT|PRIMARY', na=False) | p.str.contains('ROOT|PRIMARY|CAUSE', na=False)
    strict_mask = base_mask
    if 'CORRELATION_ENABLE' in alib.columns:
        strict_mask = base_mask & (~alib['CORRELATION_ENABLE'].astype(str).str.upper().isin(['FALSE','0','N','NO']))

    def keys(mask):
        s = set()
        for col in ['ALARM_IDENTIFIER','ALARM_ID','CLIENT_ALARM_CODE']:
            if col in alib.columns:
                s.update(alib.loc[mask, col].dropna().astype(str).tolist())
        return s

    strict_keys = keys(strict_mask)
    relax_keys = keys(base_mask)
    alarm_code = alarm['ALARM_CODE'].astype(str)
    overlap = strict_keys & set(alarm_code.unique())
    if not overlap:
        overlap = relax_keys & set(alarm_code.unique())

    if overlap:
        m = alarm[alarm_code.isin(overlap) & alarm['INCIDENT_ID'].notna() & (alarm['INCIDENT_ID'].astype(str) != 'UNKNOWN')].copy()
        m = m.sort_values('OPEN_TIME').drop_duplicates('INCIDENT_ID', keep='first')
        m['ROOT_NE_ID'] = m['ENTITY_ID_STR']
        m['ROOT_NE_NAME'] = m['ROOT_NE_ID'].map(neid_to_name)
        labels_m1 = m[['INCIDENT_ID','ROOT_NE_ID','ROOT_NE_NAME']].copy()
        labels_m1['LABEL_SOURCE'] = 'M1_alarm_library'

print('M1 labels:', len(labels_m1))

In [ ]:
# 4) Methods 2-5
# M2: earliest alarm per incident (baseline)
labels_m2 = alarm[alarm['INCIDENT_ID'].notna() & (alarm['INCIDENT_ID'].astype(str) != 'UNKNOWN')].sort_values('OPEN_TIME').drop_duplicates('INCIDENT_ID', keep='first').copy()
labels_m2['ROOT_NE_ID'] = labels_m2['ENTITY_ID_STR']
labels_m2['ROOT_NE_NAME'] = labels_m2['ROOT_NE_ID'].map(neid_to_name)
labels_m2 = labels_m2[['INCIDENT_ID','ROOT_NE_ID','ROOT_NE_NAME']]
labels_m2['LABEL_SOURCE'] = 'M2_incident_id'

# M3: CKT bridge -> NETWORK_ELEMENT_ID_FK -> network_element.ID -> NE_ID
labels_m3 = pd.DataFrame(columns=['INCIDENT_ID','ROOT_NE_ID','ROOT_NE_NAME','LABEL_SOURCE'])
if not ckt.empty and {'CKT_ID','NETWORK_ELEMENT_ID_FK'}.issubset(set(ckt.columns)):
    ckt2 = ckt.copy()
    ckt2['CKT_N'] = ckt2['CKT_ID'].map(norm_ckt)
    m3 = alarm.merge(ckt2[['CKT_N','NETWORK_ELEMENT_ID_FK']], on='CKT_N', how='inner')
    m3 = m3[m3['INCIDENT_ID'].notna() & (m3['INCIDENT_ID'].astype(str) != 'UNKNOWN')].copy()
    m3['ROOT_NE_ID'] = pd.to_numeric(m3['NETWORK_ELEMENT_ID_FK'], errors='coerce').map(id_to_neid).fillna(m3['NETWORK_ELEMENT_ID_FK'].astype(str))
    m3['ROOT_NE_NAME'] = m3['ROOT_NE_ID'].map(neid_to_name)
    m3 = m3.sort_values('OPEN_TIME').drop_duplicates('INCIDENT_ID', keep='first')
    labels_m3 = m3[['INCIDENT_ID','ROOT_NE_ID','ROOT_NE_NAME']].copy()
    labels_m3['LABEL_SOURCE'] = 'M3_ckt_bridge'

# M4: service down events -> nearest prior alarm on same CKT within lookback
labels_m4 = pd.DataFrame(columns=['INCIDENT_ID','ROOT_NE_ID','ROOT_NE_NAME','LABEL_SOURCE'])
if not ns_hist.empty and {'STATUS','MODIFIED_TIME','CKT_ID'}.issubset(set(ns_hist.columns)):
    de = ns_hist[ns_hist['STATUS'].astype(str).str.upper().str.contains('DOWN|INACTIVE|DEACTIVE|DISABLE|FAULT', na=False)].copy()
    de = de.dropna(subset=['MODIFIED_TIME','CKT_ID'])
    de['CKT_N'] = de['CKT_ID'].map(norm_ckt)
    de = de.sort_values(['CKT_N','MODIFIED_TIME'])
    al = alarm.dropna(subset=['OPEN_TIME','CKT_N']).sort_values(['CKT_N','OPEN_TIME'])
    al = al.loc[al['INCIDENT_ID'].notna() & (al['INCIDENT_ID'].astype(str) != 'UNKNOWN') & al['ENTITY_ID_STR'].notna()].copy()

    chunks = []
    tol = pd.Timedelta(minutes=120)
    for cktn, g in de.groupby('CKT_N', sort=False):
        ag = al[al['CKT_N'] == cktn]
        if ag.empty:
            continue
        x = pd.merge_asof(g[['MODIFIED_TIME']].copy().sort_values('MODIFIED_TIME'), ag.sort_values('OPEN_TIME'), left_on='MODIFIED_TIME', right_on='OPEN_TIME', direction='backward', tolerance=tol)
        chunks.append(x)
    m4 = pd.concat(chunks, ignore_index=True) if chunks else pd.DataFrame()
    if not m4.empty:
        m4 = m4[m4['INCIDENT_ID'].notna() & (m4['INCIDENT_ID'].astype(str) != 'UNKNOWN') & m4['ENTITY_ID_STR'].notna()].copy()
        m4['ROOT_NE_ID'] = m4['ENTITY_ID_STR']
        m4['ROOT_NE_NAME'] = m4['ROOT_NE_ID'].map(neid_to_name)
        m4 = m4.sort_values('OPEN_TIME').drop_duplicates('INCIDENT_ID', keep='first')
        labels_m4 = m4[['INCIDENT_ID','ROOT_NE_ID','ROOT_NE_NAME']].copy()
        labels_m4['LABEL_SOURCE'] = 'M4_service_history'

# M5: lsp_hop ROUTER_NE_ID canonicalized via network_element.ID -> NE_ID, match alarm entity
labels_m5 = pd.DataFrame(columns=['INCIDENT_ID','ROOT_NE_ID','ROOT_NE_NAME','LABEL_SOURCE'])
if not lsp_hop.empty and 'ROUTER_NE_ID' in lsp_hop.columns:
    hop = lsp_hop.copy()
    hop['ROUTER_CANON'] = pd.to_numeric(hop['ROUTER_NE_ID'], errors='coerce').map(id_to_neid).fillna(hop['ROUTER_NE_ID'].astype(str))
    if 'HOP_SEQUENCE' in hop.columns and 'LSP_ID' in hop.columns:
        hop = hop.sort_values('HOP_SEQUENCE').drop_duplicates(subset=['LSP_ID','ROUTER_CANON'], keep='first')
    al2 = alarm[['INCIDENT_ID','OPEN_TIME','ENTITY_ID_STR']].dropna(subset=['INCIDENT_ID']).copy()
    al2 = al2.sort_values('OPEN_TIME').drop_duplicates('INCIDENT_ID', keep='first')
    al2['ENTITY_CANON'] = pd.to_numeric(al2['ENTITY_ID_STR'], errors='coerce').map(id_to_neid).fillna(al2['ENTITY_ID_STR'])
    m5 = hop.merge(al2, left_on='ROUTER_CANON', right_on='ENTITY_CANON', how='inner')
    if not m5.empty and 'HOP_SEQUENCE' in m5.columns:
        m5 = m5.sort_values(['INCIDENT_ID','HOP_SEQUENCE','OPEN_TIME']).drop_duplicates('INCIDENT_ID', keep='first')
        m5['ROOT_NE_ID'] = m5['ROUTER_CANON']
        m5['ROOT_NE_NAME'] = m5['ROOT_NE_ID'].map(neid_to_name)
        labels_m5 = m5[['INCIDENT_ID','ROOT_NE_ID','ROOT_NE_NAME']].copy()
        labels_m5['LABEL_SOURCE'] = 'M5_lsp_hop'

print('M2/M3/M4/M5:', len(labels_m2), len(labels_m3), len(labels_m4), len(labels_m5))

In [ ]:
# 5) Combine labels with priority + agreement
frames = [labels_m1, labels_m2, labels_m3, labels_m4, labels_m5]
frames = [f for f in frames if f is not None and not f.empty]

if frames:
    comb = pd.concat(frames, ignore_index=True)
    priority = {'M1_alarm_library': 1, 'M2_incident_id': 2, 'M3_ckt_bridge': 3, 'M4_service_history': 4, 'M5_lsp_hop': 5}
    comb['PRIORITY'] = comb['LABEL_SOURCE'].map(priority)
    comb = comb.sort_values(['INCIDENT_ID','PRIORITY'])
    labels_df = comb.drop_duplicates('INCIDENT_ID', keep='first').reset_index(drop=True)
    agree = comb.groupby('INCIDENT_ID')['ROOT_NE_ID'].nunique()
    labels_df['MULTI_METHOD_AGREE'] = labels_df['INCIDENT_ID'].map(lambda x: agree.get(x, 1) == 1)
    os.makedirs('data_pipeline', exist_ok=True)
    labels_df.to_csv('data_pipeline/labels.csv', index=False)
    print('Total labeled incidents:', len(labels_df))
    print('Agreement:', f"{labels_df['MULTI_METHOD_AGREE'].mean():.1%}")
    print('\nLabel sources in combined candidates:')
    print(comb['LABEL_SOURCE'].value_counts())
else:
    labels_df = pd.DataFrame()
    print('No labels generated')